# 2025년 이전 종료 미연결 OpenID 검산

전수 판정표의 대상 완전성, 키 유일성, 분류 및 안전 처리 규칙을 재검산한다.

In [1]:
import csv
from collections import Counter
from pathlib import Path

root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
identity_path = root / 'data/processed/edss_0101_kedi_openid_identity_2009_2025.csv'
review_2025_path = root / 'data/metadata/edss_2025_unmatched_openid_manual_review.csv'
review_path = root / 'data/metadata/edss_pre2025_unmatched_openid_manual_review.csv'

def read_rows(path):
    with path.open(encoding='utf-8-sig', newline='') as handle:
        return list(csv.DictReader(handle))

identity = read_rows(identity_path)
reviewed_2025 = {row['open_id'] for row in read_rows(review_2025_path)}
review = read_rows(review_path)
target_ids = {row['openid'] for row in identity if row['identity_status'] == 'unmatched'} - reviewed_2025
review_ids = [row['open_id'] for row in review]
validation = {
    'target_count': len(target_ids),
    'review_count': len(review),
    'unique_review_ids': len(set(review_ids)),
    'missing_ids': sorted(target_ids - set(review_ids)),
    'extra_ids': sorted(set(review_ids) - target_ids),
    'duplicate_ids': sorted({value for value, count in Counter(review_ids).items() if count > 1}),
    'review_order_is_contiguous': [int(row['review_order']) for row in review] == list(range(1, len(review) + 1)),
    'blank_classifications': sum(not row['manual_classification'] for row in review),
    'blank_safe_actions': sum(not row['safe_join_action'] for row in review),
}
validation

{'target_count': 84,
 'review_count': 84,
 'unique_review_ids': 84,
 'missing_ids': [],
 'extra_ids': [],
 'duplicate_ids': [],
 'review_order_is_contiguous': True,
 'blank_classifications': 0,
 'blank_safe_actions': 0}

In [2]:
classification_counts = Counter(row['manual_classification'] for row in review)
candidate_status_counts = Counter(row['candidate_status'] for row in review)
{
    'classification_counts': dict(sorted(classification_counts.items())),
    'candidate_status_counts': dict(sorted(candidate_status_counts.items())),
    'named_candidate_count': sum(bool(row['likely_entity_candidate']) for row in review),
    'approved_identity_count': candidate_status_counts['confirmed_manual_reviewed_kedi_identity'],
    'final_identity_count': candidate_status_counts['confirmed_final_manual_review_identity'],
    'unconfirmed_status_count': sum(not status.startswith('confirmed_') for status in candidate_status_counts),
    'closed_without_name_count': sum(
        row['manual_classification'] == 'closed_school_or_department_residual_id'
        and not row['likely_entity_candidate'] for row in review
    ),
}

{'classification_counts': {'confirmed_identity_closed_campus_conversion_predecessor_id': 1,
  'confirmed_identity_closed_campus_residual_id': 1,
  'confirmed_identity_closed_name_change_predecessor_id': 12,
  'confirmed_identity_closed_school_residual_id': 54,
  'confirmed_identity_department_residual_id': 4,
  'confirmed_identity_historical_reorganization_predecessor_id': 3,
  'confirmed_identity_institution_type_conversion_predecessor_id': 1,
  'confirmed_identity_merger_residual_predecessor_id': 6,
  'confirmed_identity_name_change_predecessor_id': 2},
 'candidate_status_counts': {'confirmed_exact_longitudinal_kedi_name_metrics_and_closed_departments': 1,
  'confirmed_exact_longitudinal_metrics_and_closed_status': 1,
  'confirmed_exact_longitudinal_metrics_and_official_name_change': 1,
  'confirmed_exact_metric_and_official_history': 1,
  'confirmed_final_graduation_department_and_official_closure': 1,
  'confirmed_final_graduation_department_and_official_conversion': 1,
  'confirme

In [3]:
assert validation['target_count'] == 84
assert validation['review_count'] == 84
assert validation['unique_review_ids'] == 84
assert not validation['missing_ids']
assert not validation['extra_ids']
assert not validation['duplicate_ids']
assert validation['review_order_is_contiguous']
assert validation['blank_classifications'] == 0
assert validation['blank_safe_actions'] == 0
assert classification_counts == {
    'confirmed_identity_closed_campus_residual_id': 1,
    'confirmed_identity_closed_campus_conversion_predecessor_id': 1,
    'confirmed_identity_closed_name_change_predecessor_id': 12,
    'confirmed_identity_closed_school_residual_id': 54,
    'confirmed_identity_department_residual_id': 4,
    'confirmed_identity_historical_reorganization_predecessor_id': 3,
    'confirmed_identity_institution_type_conversion_predecessor_id': 1,
    'confirmed_identity_merger_residual_predecessor_id': 6,
    'confirmed_identity_name_change_predecessor_id': 2,
}
assert candidate_status_counts['confirmed_manual_reviewed_kedi_identity'] == 30
assert candidate_status_counts['confirmed_final_manual_review_identity'] == 16
assert all(status.startswith('confirmed_') for status in candidate_status_counts)
assert all(row['likely_entity_candidate'] for row in review)
'all checks passed'

'all checks passed'